# Supplement — Training the Week 2 Classifiers with PyTorch

**CSS371: Natural Language Processing**  
**Purpose:** reproduce the Week 2 from-scratch tasks with a framework and compare the training mechanics

This notebook uses the same mock datasets and model structures as `week02_neural_networks_optimization.ipynb`, but replaces manually implemented derivatives and updates with:

- `torch.nn` layers;
- `torch.nn.BCEWithLogitsLoss`;
- `loss.backward()` automatic differentiation;
- `torch.optim.SGD` parameter updates.

PyTorch uses **automatic differentiation**, not finite-difference numerical gradients, during ordinary training.


## Learning outcomes

Students should be able to:

1. map each from-scratch operation to its PyTorch equivalent;
2. explain the roles of `requires_grad`, computation graphs, `backward`, and optimizers;
3. distinguish logits from probabilities;
4. explain why `BCEWithLogitsLoss` combines sigmoid and BCE;
5. verify PyTorch gradients against the manually derived matrix formula;
6. train linear, nonlinear, and text classifiers;
7. identify what PyTorch automates and what remains the practitioner's responsibility.


In [3]:
import sys, random, re
from collections import Counter
import numpy as np
import torch
from torch import nn

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | PyTorch {torch.__version__} | {DEVICE}")


Python 3.12.7 | NumPy 1.26.4 | PyTorch 2.13.0+cpu | cpu


## 1. From-scratch versus PyTorch vocabulary

| From-scratch NumPy | PyTorch |
|---|---|
| parameter arrays `W`, `b` | `nn.Parameter` inside a module |
| `Z = X @ W + b` | `nn.Linear` |
| manual sigmoid + BCE | `BCEWithLogitsLoss` |
| manual cache | dynamic computation graph |
| `dZ`, `dW`, `db` code | `loss.backward()` |
| `parameter -= lr * gradient` | `optimizer.step()` |
| manually clear arrays if reused | `optimizer.zero_grad()` |

PyTorch automates differentiation and tensor kernels. It does **not** choose the data, representation, architecture, loss, split, metric, learning rate, or stopping criterion for us.


# Part I — Linear binary classification

## 2. Recreate exactly the same mock data

The NumPy generator and seed match the main Week 2 notebook. We convert arrays to `float32` tensors after the train/validation split.


In [6]:
def make_blob_data(n_per_class=100, noise=0.75, seed=42):
    rng=np.random.default_rng(seed)
    c0=rng.normal(loc=(-1.0,-1.0),scale=noise,size=(n_per_class,2))
    c1=rng.normal(loc=(1.0,1.0),scale=noise,size=(n_per_class,2))
    X=np.vstack([c0,c1]); y=np.vstack([np.zeros((n_per_class,1)),np.ones((n_per_class,1))])
    order=rng.permutation(len(X)); return X[order],y[order]

X_np,y_np=make_blob_data(); split=int(.8*len(X_np))
X_train=torch.tensor(X_np[:split],dtype=torch.float32,device=DEVICE)
y_train=torch.tensor(y_np[:split],dtype=torch.float32,device=DEVICE)
X_valid=torch.tensor(X_np[split:],dtype=torch.float32,device=DEVICE)
y_valid=torch.tensor(y_np[split:],dtype=torch.float32,device=DEVICE)
print(X_train.shape,y_train.shape,X_valid.shape,y_valid.shape)


torch.Size([160, 2]) torch.Size([160, 1]) torch.Size([40, 2]) torch.Size([40, 1])


## 3. `nn.Linear(2,1)` is the same affine model

PyTorch stores the linear-layer weight as shape `(out_features, in_features)`:

$$W_{torch}:(1,2),\qquad b:(1).$$

Its forward equation is equivalent to

$$Z=XW_{manual}+b=XW_{torch}^{\top}+b.$$

The layer returns logits. `BCEWithLogitsLoss` internally evaluates BCE using a stable log-sum-exp formulation rather than explicitly calculating `log(sigmoid(z))`.


In [8]:
linear_model=nn.Linear(2,1).to(DEVICE)
with torch.no_grad():
    linear_model.weight.zero_(); linear_model.bias.zero_()
criterion=nn.BCEWithLogitsLoss()
optimizer=torch.optim.SGD(linear_model.parameters(),lr=0.1)

print("weight shape:",tuple(linear_model.weight.shape))
print("bias shape:  ",tuple(linear_model.bias.shape))
print("parameters require gradients:",[p.requires_grad for p in linear_model.parameters()])


weight shape: (1, 2)
bias shape:   (1,)
parameters require gradients: [True, True]


## 4. Verify autograd against the analytical formula

For mean BCE with sigmoid output $P=\sigma(Z)$:

$$dZ=\frac{P-Y}{N},\qquad dW_{manual}=X^\top dZ,\qquad db=\sum_i dZ_i.$$

After `loss.backward()`, PyTorch stores gradients in `.grad`. Because PyTorch's weight is transposed relative to the manual convention:

$$W_{torch}.grad=(dW_{manual})^\top.$$


In [10]:
optimizer.zero_grad()
logits=linear_model(X_train)
loss=criterion(logits,y_train)
loss.backward()

with torch.no_grad():
    probabilities=torch.sigmoid(logits)
    dZ=(probabilities-y_train)/len(X_train)
    manual_dW=X_train.T@dZ
    manual_db=dZ.sum(dim=0)

print("autograd weight gradient:",linear_model.weight.grad)
print("manual dW transposed:     ",manual_dW.T)
print("autograd bias gradient:   ",linear_model.bias.grad)
print("manual db:                ",manual_db)
print("maximum weight difference:",(linear_model.weight.grad-manual_dW.T).abs().max().item())
assert torch.allclose(linear_model.weight.grad,manual_dW.T,atol=1e-7)
assert torch.allclose(linear_model.bias.grad,manual_db,atol=1e-7)


autograd weight gradient: tensor([[-0.5083, -0.5055]])
manual dW transposed:      tensor([[-0.5083, -0.5055]])
autograd bias gradient:    tensor([0.0188])
manual db:                 tensor([0.0188])
maximum weight difference: 0.0


## 5. The standard PyTorch training loop

```text
zero_grad → forward → loss → backward → optimizer.step
```

1. `zero_grad()` clears accumulated gradients.
2. Forward operations construct a computation graph.
3. Loss produces one scalar objective.
4. `backward()` applies reverse-mode automatic differentiation.
5. `step()` updates parameters using the optimizer rule.

PyTorch accumulates gradients by default because accumulation is useful for some workflows. Forgetting to clear them unintentionally changes the update.


In [12]:
# Reset to zero so results align with the from-scratch experiment.
with torch.no_grad(): linear_model.weight.zero_(); linear_model.bias.zero_()
linear_history=[]
for epoch in range(401):
    optimizer.zero_grad()
    logits=linear_model(X_train)
    loss=criterion(logits,y_train)
    loss.backward()
    optimizer.step()
    linear_history.append(loss.item())
    if epoch%100==0:
        with torch.no_grad():
            accuracy=((torch.sigmoid(logits)>=.5)==y_train.bool()).float().mean().item()
        print(f"epoch={epoch:3d} loss={loss.item():.4f} train_accuracy={accuracy:.3f}")

linear_model.eval()
with torch.no_grad():
    valid_probability=torch.sigmoid(linear_model(X_valid))
    valid_accuracy=((valid_probability>=.5)==y_valid.bool()).float().mean().item()
print("learned weight:",linear_model.weight.detach().cpu().numpy())
print("learned bias:  ",linear_model.bias.detach().cpu().numpy())
print("validation accuracy:",valid_accuracy)
assert linear_history[-1]<linear_history[0]


epoch=  0 loss=0.6931 train_accuracy=0.481
epoch=100 loss=0.1308 train_accuracy=0.981
epoch=200 loss=0.1011 train_accuracy=0.981
epoch=300 loss=0.0900 train_accuracy=0.981
epoch=400 loss=0.0841 train_accuracy=0.981
learned weight: [[2.1433542 2.1442292]]
learned bias:   [-0.00166671]
validation accuracy: 1.0


### Expected comparison with NumPy

With identical zero initialization, full-batch SGD, mean BCE, learning rate, precision close enough, and update count, the learned parameters should closely match the from-scratch model. Minor differences can arise from `float32` versus NumPy `float64` and implementation details.


# Part II — Nonlinear XOR classification

## 6. Replace manual matrices with modules

The from-scratch model used

$$XW_1+b_1\rightarrow\operatorname{ReLU}\rightarrow A_1W_2+b_2.$$

PyTorch expresses the same graph with `nn.Sequential`. The final layer returns logits; no explicit sigmoid is placed inside the model during training because the loss already incorporates it.


In [15]:
def make_xor_data(n=600,noise=.18,seed=7):
    rng=np.random.default_rng(seed); X=rng.uniform(-1,1,size=(n,2))
    score=X[:,0]*X[:,1]+rng.normal(0,noise,size=n)
    y=(score<0).astype(float).reshape(-1,1); order=rng.permutation(n)
    return X[order],y[order]

X_xor_np,y_xor_np=make_xor_data(); cut=480
X_xor_train=torch.tensor(X_xor_np[:cut],dtype=torch.float32,device=DEVICE)
y_xor_train=torch.tensor(y_xor_np[:cut],dtype=torch.float32,device=DEVICE)
X_xor_valid=torch.tensor(X_xor_np[cut:],dtype=torch.float32,device=DEVICE)
y_xor_valid=torch.tensor(y_xor_np[cut:],dtype=torch.float32,device=DEVICE)

torch.manual_seed(11)
xor_model=nn.Sequential(nn.Linear(2,12),nn.ReLU(),nn.Linear(12,1)).to(DEVICE)
xor_optimizer=torch.optim.SGD(xor_model.parameters(),lr=.15)
print(xor_model)


Sequential(
  (0): Linear(in_features=2, out_features=12, bias=True)
  (1): ReLU()
  (2): Linear(in_features=12, out_features=1, bias=True)
)


In [16]:
xor_history=[]
for epoch in range(2001):
    xor_optimizer.zero_grad()
    logits=xor_model(X_xor_train)
    loss=criterion(logits,y_xor_train)
    loss.backward()
    xor_optimizer.step()
    xor_history.append(loss.item())
    if epoch%500==0: print(f"epoch={epoch:4d} loss={loss.item():.4f}")

xor_model.eval()
with torch.no_grad():
    xor_probability=torch.sigmoid(xor_model(X_xor_valid))
    xor_accuracy=((xor_probability>=.5)==y_xor_valid.bool()).float().mean().item()
print("XOR validation accuracy:",xor_accuracy)
assert xor_history[-1]<xor_history[0]


epoch=   0 loss=0.7046
epoch= 500 loss=0.4160
epoch=1000 loss=0.3958
epoch=1500 loss=0.3945
epoch=2000 loss=0.3936
XOR validation accuracy: 0.7583333253860474


## 7. Inspect gradients by layer

Autograd calculates one gradient tensor for every trainable parameter. The gradient shape must equal its parameter shape. Gradient norms help diagnose vanishing, exploding, or inactive paths, though a norm alone does not establish correctness.


In [18]:
xor_model.train(); xor_optimizer.zero_grad()
small_loss=criterion(xor_model(X_xor_train[:16]),y_xor_train[:16]); small_loss.backward()
for name,parameter in xor_model.named_parameters():
    print(f"{name:8s} parameter={tuple(parameter.shape)!s:8s} gradient={tuple(parameter.grad.shape)!s:8s} norm={parameter.grad.norm().item():.6f}")
    assert parameter.shape==parameter.grad.shape


0.weight parameter=(12, 2)  gradient=(12, 2)  norm=0.138823
0.bias   parameter=(12,)    gradient=(12,)    norm=0.131655
2.weight parameter=(1, 12)  gradient=(1, 12)  norm=0.120712
2.bias   parameter=(1,)     gradient=(1,)     norm=0.048158


# Part III — Sentence classification

## 8. Keep preprocessing identical

PyTorch does not automatically turn raw sentences into meaningful model input. We reuse the Week 2 tokenizer, training-only vocabulary, and binary Bag-of-Words vectors. Only the classifier and differentiation mechanism change.


In [20]:
train_sentences=[
"the lecture was clear and useful","I enjoyed the practical examples","the explanation was easy to follow",
"excellent lesson and helpful exercises","the notebook is detailed and clear","I learned a lot from this class",
"the examples were interesting","good pacing and useful code","the lecture was confusing and slow",
"I disliked the unclear explanation","the examples were boring","poor lesson with unhelpful exercises",
"the notebook is difficult and messy","I learned nothing from this class","the code was confusing","bad pacing and unclear slides"]
train_labels_np=np.array([1]*8+[0]*8).reshape(-1,1)
valid_sentences=["clear examples and useful exercises","interesting and helpful lecture",
"confusing notebook and poor explanation","boring examples with bad pacing","the code was clear","the lesson was messy"]
valid_labels_np=np.array([1,1,0,0,1,0]).reshape(-1,1)

def tokenize_sentence(s): return re.findall(r"[a-z]+(?:'[a-z]+)?",s.lower())
def build_vocabulary(sentences):
    counts=Counter(t for s in sentences for t in tokenize_sentence(s))
    return {t:i for i,t in enumerate(sorted(counts))}
def vectorize(sentences,vocab):
    matrix=np.zeros((len(sentences),len(vocab)),dtype=np.float32)
    for row,s in enumerate(sentences):
        for token in tokenize_sentence(s):
            if token in vocab: matrix[row,vocab[token]]=1
    return matrix

vocab=build_vocabulary(train_sentences)
X_text_train=torch.tensor(vectorize(train_sentences,vocab),device=DEVICE)
y_text_train=torch.tensor(train_labels_np,dtype=torch.float32,device=DEVICE)
X_text_valid=torch.tensor(vectorize(valid_sentences,vocab),device=DEVICE)
y_text_valid=torch.tensor(valid_labels_np,dtype=torch.float32,device=DEVICE)
print("vocabulary:",len(vocab),"train tensor:",tuple(X_text_train.shape))


vocabulary: 45 train tensor: (16, 45)


In [21]:
text_model=nn.Linear(len(vocab),1).to(DEVICE)
with torch.no_grad(): text_model.weight.zero_(); text_model.bias.zero_()
text_optimizer=torch.optim.SGD(text_model.parameters(),lr=.2)
text_history=[]
for epoch in range(1001):
    text_optimizer.zero_grad()
    logits=text_model(X_text_train)
    loss=criterion(logits,y_text_train)
    loss.backward(); text_optimizer.step(); text_history.append(loss.item())
    if epoch%250==0:
        accuracy=((torch.sigmoid(logits)>=.5)==y_text_train.bool()).float().mean().item()
        print(f"epoch={epoch:4d} loss={loss.item():.4f} train_accuracy={accuracy:.3f}")

text_model.eval()
with torch.no_grad(): valid_p=torch.sigmoid(text_model(X_text_valid)).cpu().numpy().ravel()
for sentence,pred,label in zip(valid_sentences,valid_p,valid_labels_np.ravel()):
    print(f"p={pred:.3f} pred={int(pred>=.5)} true={label} | {sentence}")


epoch=   0 loss=0.6931 train_accuracy=0.500
epoch= 250 loss=0.1513 train_accuracy=1.000
epoch= 500 loss=0.0802 train_accuracy=1.000
epoch= 750 loss=0.0538 train_accuracy=1.000
epoch=1000 loss=0.0402 train_accuracy=1.000
p=0.998 pred=1 true=1 | clear examples and useful exercises
p=0.981 pred=1 true=1 | interesting and helpful lecture
p=0.049 pred=0 true=0 | confusing notebook and poor explanation
p=0.019 pred=0 true=0 | boring examples with bad pacing
p=0.765 pred=1 true=1 | the code was clear
p=0.096 pred=0 true=0 | the lesson was messy


## 9. Inspect text weights

The PyTorch layer stores weights as `(1, vocabulary_size)`. As in the from-scratch model, positive weights push the logit toward positive sentiment and negative weights toward negative sentiment.


In [23]:
weights=text_model.weight.detach().cpu().numpy().ravel(); index_to_token={i:t for t,i in vocab.items()}
order=np.argsort(weights)
print("negative:",[(index_to_token[i],round(float(weights[i]),3)) for i in order[:6]])
print("positive:",[(index_to_token[i],round(float(weights[i]),3)) for i in order[-6:][::-1]])


negative: [('boring', -2.538), ('confusing', -2.407), ('unclear', -2.181), ('nothing', -1.976), ('messy', -1.511), ('difficult', -1.511)]
positive: [('interesting', 2.281), ('clear', 2.206), ('useful', 2.016), ('lot', 1.829), ('a', 1.829), ('enjoyed', 1.381)]


## 10. What PyTorch automated

PyTorch automated:

- parameter registration;
- computation-graph construction;
- reverse-mode chain rule;
- gradient accumulation in `.grad`;
- SGD update implementation;
- optimized tensor operations and device placement.

We still decided:

- dataset and split;
- feature representation;
- architecture and hidden width;
- loss and its reduction;
- optimizer and learning rate;
- number of epochs, threshold, and metrics;
- how to investigate errors and generalization.

Framework fluency is not a substitute for understanding the derivatives. The explicit proof explains what `backward()` computes; PyTorch makes the same computation scalable and less error-prone.


## 11. Autograd versus numerical gradient

During `backward()`, PyTorch applies registered derivative rules to the computation graph. It does not perturb every parameter by $\epsilon$.

Numerical finite differences remain useful for testing custom functions. `torch.autograd.gradcheck` compares autograd against finite differences, generally using double precision and small inputs. It is a diagnostic, not the normal training method.


## Exercises

1. Remove `zero_grad()` and record gradient norms for five updates.
2. Replace SGD with momentum and compare XOR convergence.
3. Compare `BCELoss(sigmoid(logits))` with `BCEWithLogitsLoss(logits)` for logits ±100.
4. Freeze the first XOR layer by setting `requires_grad=False`; explain the result.
5. Train on mini-batches and compare the variability of loss.
6. Add bigram features to sentence classification without changing the PyTorch model.
7. Use `torch.autograd.gradcheck` on a tiny custom sigmoid+BCE function in `float64`.


## Summary

- PyTorch ordinary training uses automatic differentiation, not numerical differentiation.
- `nn.Linear` implements affine transformation; its stored weight orientation differs from the manual convention.
- `BCEWithLogitsLoss` accepts logits and combines sigmoid+BCE stably.
- Autograd gradients match $X^\top(P-Y)/N$ after accounting for weight orientation.
- `backward()` computes/accumulates gradients; `optimizer.step()` updates parameters.
- The same framework pattern trains linear, nonlinear, and Bag-of-Words sentence classifiers.
- Data design, representation, evaluation, and interpretation remain our responsibility.
